MANC notebook- create neuroglancer linsk from MANC version v1.2. Jelly Soffers 20260307

In [19]:
# set up environment
import sys
import json
import re
import requests
import pandas as pd

from caveclient import CAVEclient

print("Python executable:", sys.executable)
print("Imports OK")

Python executable: c:\Users\JHS\Documents\Python\cave_env\Scripts\python.exe
Imports OK


In [20]:
#set up a output directory for tables etc.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output:", OUTPUT_DIR)



Project root: c:\Users\JHS\Documents\Python\project_folder_4B
Output: c:\Users\JHS\Documents\Python\project_folder_4B\output


In [21]:
# work backwards from my 4B T1 clio URL to get filters (manually done in 3D viewer and excel) clearly written down in code. So that then we can compare it to FANC
import json
import urllib.parse

CLIO_URL = r"""https://clio-ng.janelia.org/#!%7B%22title%22:%224B%20connectivity%22%2C%22dimensions%22:%7B%22x%22:%5B8e-9%2C%22m%22%5D%2C%22y%22:%5B8e-9%2C%22m%22%5D%2C%22z%22:%5B8e-9%2C%22m%22%5D%7D%2C%22position%22:%5B15353.5%2C35844.5%2C53248.5%5D%2C%22crossSectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22:1%2C%22projectionOrientation%22:%5B-0.29617950320243835%2C0.698911726474762%2C-0.6356714367866516%2C0.14043490588665009%5D%2C%22projectionScale%22:26672.2540375702%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22source%22%2C%22name%22:%22em%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22toolBindings%22:%7B%22Q%22:%22selectSegments%22%7D%2C%22tab%22:%22source%22%2C%22selectedAlpha%22:0.34%2C%22saturation%22:0.49%2C%22ignoreNullVisibleSet%22:false%2C%22segments%22:%5B%2210068%22%5D%2C%22segmentQuery%22:%22//./%20#class:Sensory_TBD%20%7CPreSyn%20%7CPostSyn%22%2C%22segmentDefaultColor%22:%22#00ff80%22%2C%22segmentColors%22:%7B%2233946%22:%22#808080%22%7D%2C%22name%22:%22manc:v1.2.1-TTMn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/all-vnc-roi%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22source%22%2C%22annotationColor%22:%22#37c842%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22name%22:%22all-tissue%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/synaptic-neuropil%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22segmentColors%22:%7B%221%22:%22#ffffff%22%7D%2C%22name%22:%22all-synaptic%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/roi-202208%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22neuropils%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/nerve-roi-202301%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22objectAlpha%22:0.1%2C%22segments%22:%5B%221%22%2C%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%222%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%2228%22%2C%2229%22%2C%223%22%2C%2230%22%2C%2231%22%2C%2232%22%2C%2233%22%2C%2234%22%2C%2235%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22nerves%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPre%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPostPartners%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPre%20&&%20showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPre%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPostPartners%22:false%2C%22postColor%22:%22#00ffff%22%2C%22preConfidence%22:0.4%2C%22postConfidence%22:0.4%7D%2C%22linkedSegmentationLayer%22:%7B%22pre_synaptic_cell%22:%22manc:v1.2.1-TTMn%22%7D%2C%22filterBySegmentation%22:%5B%22pre_synaptic_cell%22%5D%2C%22name%22:%22presyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPrePartners%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPost%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPrePartners%20&&%20showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPrePartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPrePartners%22:false%2C%22postColor%22:%22#00ffff%22%7D%2C%22linkedSegmentationLayer%22:%7B%22post_synaptic_cell%22:%22manc:v1.2.1-TTMn%22%7D%2C%22filterBySegmentation%22:%5B%22post_synaptic_cell%22%5D%2C%22name%22:%22postsyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/mask%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22segmentColors%22:%7B%221%22:%22#000000%22%7D%2C%22name%22:%22voxel-classes%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/rc5_wsexp%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22segments%22%2C%22name%22:%22initial-supervoxels%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/court-et-al-systematic-manc_tracts/mesh%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22rendering%22%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%22100%22%2C%22101%22%2C%22102%22%2C%22103%22%2C%22104%22%2C%22105%22%2C%22106%22%5D%2C%22name%22:%22court%20et%20al.%20tracts%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22objectAlpha%22:0.03%2C%22hideSegmentZero%22:false%2C%22segments%22:%5B%2210024%22%2C%22100339%22%2C%22100525%22%2C%22101447%22%2C%22101453%22%2C%22102138%22%2C%22102536%22%2C%2210314%22%2C%2211143%22%2C%2212523%22%2C%2213128%22%2C%2213409%22%2C%2213436%22%2C%2213455%22%2C%2213506%22%2C%2213589%22%2C%2213770%22%2C%2213798%22%2C%2213975%22%2C%2214024%22%2C%2214031%22%2C%2214389%22%2C%2214697%22%2C%2214728%22%2C%2214774%22%2C%2214954%22%2C%2215002%22%2C%2215005%22%2C%22152244%22%2C%22152655%22%2C%2215309%22%2C%22153878%22%2C%2215393%22%2C%2215401%22%2C%22155236%22%2C%2215811%22%2C%22158391%22%2C%2215969%22%2C%2216224%22%2C%22162540%22%2C%2216293%22%2C%22164102%22%2C%22164146%22%2C%2216487%22%2C%2216527%22%2C%22165318%22%2C%22166421%22%2C%2216757%22%2C%2217351%22%2C%2217395%22%2C%2217409%22%2C%2217490%22%2C%2217534%22%2C%2217567%22%2C%2217572%22%2C%2217678%22%2C%2217935%22%2C%2218148%22%2C%2218156%22%2C%2218253%22%2C%2218571%22%2C%2218605%22%2C%2218629%22%2C%2218641%22%2C%2218724%22%2C%2218755%22%2C%2218785%22%2C%2218831%22%2C%2218990%22%2C%2219250%22%2C%2219366%22%2C%2219833%22%2C%2219912%22%2C%2219985%22%2C%2220077%22%2C%2220585%22%2C%2220836%22%2C%2220932%22%2C%2221189%22%2C%2221322%22%2C%2221499%22%2C%2221567%22%2C%2221808%22%2C%2221862%22%2C%2221956%22%2C%2222163%22%2C%2222868%22%2C%2222997%22%2C%2223716%22%2C%2223790%22%2C%2224181%22%2C%2226388%22%2C%2226941%22%2C%2227760%22%2C%2229988%22%2C%2236722%22%2C%2238071%22%5D%2C%22segmentQuery%22:%22#hemilineage:04B%20#somaNeuromere:T1%20#somaSide:LHS%22%2C%22segmentDefaultColor%22:%22#e9e2e2%22%2C%22name%22:%22manc-seg-v1.2-4B-all%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22objectAlpha%22:0.18%2C%22segments%22:%5B%22152244%22%2C%2215309%22%2C%2218148%22%2C%2219250%22%2C%2219366%22%2C%2221956%22%5D%2C%22segmentQuery%22:%22#hemilineage:04B%20#somaSide:LHS%20#somaNeuromere:T1%20#subclass:BR%22%2C%22segmentDefaultColor%22:%22#ff0000%22%2C%22name%22:%22manc-seg-v1.21-4B-subclassBR%20n=26%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0.4%2C%22objectAlpha%22:0.47%2C%22segments%22:%5B%2210314%22%2C%2211143%22%2C%2213436%22%2C%2214031%22%2C%2215393%22%2C%2215401%22%2C%2215969%22%2C%2216293%22%2C%2217395%22%2C%2219912%22%5D%2C%22segmentQuery%22:%22#hemilineage:04B%20#somaNeuromere:T1%20#somaSide:LHS%20#subclass:BI%22%2C%22segmentDefaultColor%22:%22#88bf98%22%2C%22name%22:%22manc-seg-v1.2-4B-subbclass%20BI%20n=4%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22objectAlpha%22:0.61%2C%22linkedSegmentationColorGroup%22:%22neuropils%22%2C%22segments%22:%5B%2213409%22%5D%2C%22name%22:%22manc-seg-v1.2-4B-sublass%20IR%20n=60%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2217490%22%5D%2C%22segmentQuery%22:%22#hemilineage:04B%20#somaNeuromere:T1%20#somaSide:LHS%20#subclass:BA%22%2C%22name%22:%22manc-seg-v1.21-4B%20subclass%20BA%20n=1%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2210314%22%2C%2213436%22%2C%2217395%22%2C%2219912%22%5D%2C%22segmentQuery%22:%22#hemilineage:04B%20#somaNeuromere:T1%20#somaSide:LHS%20#subclass:IA%22%2C%22segmentDefaultColor%22:%22#dd7878%22%2C%22name%22:%22manc-seg-v1.22-4B%20subclass%20IA=4%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22rendering%22%2C%22objectAlpha%22:0.41%2C%22segments%22:%5B%22101453%22%2C%2210314%22%2C%2214389%22%2C%2215002%22%2C%2215309%22%2C%22155236%22%2C%2215811%22%2C%2215969%22%2C%22162540%22%2C%2216757%22%2C%2217351%22%2C%2217490%22%2C%2217572%22%2C%2217678%22%2C%2218785%22%2C%2218831%22%2C%2219912%22%2C%2220077%22%2C%2221567%22%2C%2221956%22%5D%2C%22segmentQuery%22:%22#hemilineage:04B%20#somaNeuromere:T1%20#somaSide:LHS%20#birthtime:early_secondary%22%2C%22segmentDefaultColor%22:%22#fff700%22%2C%22name%22:%22manc-seg-v1.22-4B-early%20secondary%20n=%2020%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22rendering%22%2C%22objectAlpha%22:0.35%2C%22segments%22:%5B%2210024%22%2C%22100339%22%2C%22100525%22%2C%22101447%22%2C%22102138%22%2C%22102536%22%2C%2213128%22%2C%2213409%22%2C%2213455%22%2C%2213506%22%2C%2213589%22%2C%2213770%22%2C%2213798%22%2C%2213975%22%2C%2214024%22%2C%2214031%22%2C%2214697%22%2C%2214728%22%2C%2214774%22%2C%2214954%22%2C%2215005%22%2C%22152244%22%2C%22152655%22%2C%22153878%22%2C%2215393%22%2C%2215401%22%2C%22158391%22%2C%2216224%22%2C%2216293%22%2C%22164102%22%2C%22164146%22%2C%2216487%22%2C%2216527%22%2C%22165318%22%2C%22166421%22%2C%2217409%22%2C%2217534%22%2C%2217567%22%2C%2217935%22%2C%2218148%22%2C%2218156%22%2C%2218253%22%2C%2218571%22%2C%2218605%22%2C%2218629%22%2C%2218641%22%2C%2218724%22%2C%2218755%22%2C%2218990%22%2C%2219250%22%2C%2219366%22%2C%2219833%22%2C%2219985%22%2C%2220585%22%2C%2220836%22%2C%2220932%22%2C%2221189%22%2C%2221322%22%2C%2221499%22%2C%2221808%22%2C%2221862%22%2C%2222163%22%2C%2222868%22%2C%2222997%22%2C%2223716%22%2C%2223790%22%2C%2224181%22%2C%2226388%22%2C%2226941%22%2C%2227760%22%2C%2229988%22%2C%2236722%22%2C%2238071%22%5D%2C%22segmentQuery%22:%22#hemilineage:04B%20#somaNeuromere:T1%20#somaSide:LHS%20#birthtime:secondary%22%2C%22segmentDefaultColor%22:%22#a1ddde%22%2C%22name%22:%22manc-seg-v1.21-%20secondary%20n=73%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2211143%22%2C%2212523%22%2C%2213436%22%2C%2217395%22%5D%2C%22segmentQuery%22:%22#hemilineage:04B%20#somaNeuromere:T1%20#somaSide:LHS%20#birthtime:primary%22%2C%22segmentDefaultColor%22:%22#00ccff%22%2C%22name%22:%22manc-seg-v1.2%204B-%20primary%20%20n=4%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22rendering%22%2C%22objectAlpha%22:0.36%2C%22segments%22:%5B%22152244%22%2C%2215309%22%2C%2218148%22%2C%2219250%22%2C%2219366%22%2C%2221956%22%5D%2C%22segmentQuery%22:%22%2C%2018148%2019250%2019366%20%2C%20%2C%2015309%2C%20152244%2021956%22%2C%22segmentDefaultColor%22:%22#ff0000%22%2C%22name%22:%22manc-seg-v1.2%204b-manual%20BR-4%20n=6%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22objectAlpha%22:0.64%2C%22segments%22:%5B%2215002%22%2C%22155236%22%2C%22164102%22%2C%2218755%22%5D%2C%22segmentQuery%22:%22%20%2C%20155236%20164102%2015002%2C%20%2018755%2C%22%2C%22segmentDefaultColor%22:%22#e6517e%22%2C%22name%22:%22manc-seg-v1.2%204B%20manual%20BR-3%20n=4%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22rendering%22%2C%22objectAlpha%22:0.51%2C%22segments%22:%5B%2210024%22%2C%22101447%22%2C%2212523%22%2C%22162540%22%2C%2216487%22%2C%2216527%22%2C%2217351%22%2C%2217409%22%2C%2217534%22%2C%2218571%22%5D%2C%22segmentQuery%22:%2212523%20162540%2016527%2010024%2016487%2017351%2017409%2018571%2017534%20101447%22%2C%22segmentDefaultColor%22:%22#be2727%22%2C%22name%22:%22manc-seg-v1.2%204B%20manual%20BR-2%20n=10%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22source%22%2C%22objectAlpha%22:0.98%2C%22segments%22:%5B%2213455%22%2C%2213506%22%2C%2213589%22%2C%2214728%22%2C%2214774%22%2C%2215005%22%5D%2C%22segmentQuery%22:%2213455%2013506%2013589%2014728%2014774%2015005%22%2C%22segmentDefaultColor%22:%22#d75c09%22%2C%22name%22:%22manc-seg-v1.2%204B%20manual%20BR-1%20n=6%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22source%22%2C%22segments%22:%5B%22100339%22%2C%2216224%22%2C%2220836%22%2C%2226388%22%5D%2C%22segmentQuery%22:%2216224%2020836%20100339%2018831%2026388%2021808%2020932%2036722%20158391%2023790%2020585%22%2C%22name%22:%22manc-seg-v1.2%204B%20manual%20IR-independent%20leg%20n=11%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22source%22%2C%22segments%22:%5B%22152655%22%2C%22166421%22%2C%2221862%22%5D%2C%22segmentQuery%22:%2221862%20152655%20166421%22%2C%22name%22:%22manc-seg-v1.2%204B%20manual%20%20IR-%20proprioceptice%20tactile%20n=3%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22101453%22%2C%2217567%22%2C%2217572%22%2C%2218156%22%2C%2218785%22%2C%2220077%22%2C%2221567%22%5D%2C%22segmentQuery%22:%2220077%2021567%2017567%2018156%20101453%2017572%2018785%22%2C%22name%22:%22manc-seg-v1.2%204B%20manual%20%20IR-nomorphology%20assigned-early%20secondary%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22164146%22%2C%2218156%22%2C%2222868%22%2C%2223716%22%2C%2226941%22%2C%2227760%22%2C%2238071%22%5D%2C%22name%22:%22manc-seg-v1.2%204B%20manual%20IR%20nomorphology%20assigned%20secondary%22%7D%5D%2C%22showSlices%22:false%2C%22prefetch%22:false%2C%22selectedLayer%22:%7B%22flex%22:1.49%2C%22visible%22:true%2C%22layer%22:%22manc-seg-v1.2%204B%20manual%20IR%20nomorphology%20assigned%20secondary%22%7D%2C%22crossSectionBackgroundColor%22:%22#808080%22%2C%22projectionBackgroundColor%22:%22#ffffff%22%2C%22layout%22:%223d%22%2C%22selection%22:%7B%22flex%22:0.51%2C%22position%22:%5B35430.42578125%2C35617.8828125%2C36043.18359375%5D%2C%22layers%22:%7B%22all-tissue%22:%7B%22value%22:%221%22%7D%7D%7D%2C%22layerListPanel%22:%7B%22visible%22:true%7D%7D"""

# Split off the encoded JSON fragment after "#!"
if "#!" not in CLIO_URL:
    raise ValueError("Expected a clio-ng neuroglancer link containing '#!'")

encoded = CLIO_URL.split("#!", 1)[1]

# URL-decode into a JSON string
json_str = urllib.parse.unquote(encoded)

# Parse JSON into a Python dict
state = json.loads(json_str)

print("Top-level keys:", list(state.keys()))
print("Number of layers:", len(state.get("layers", [])))

# Save a local copy (this is your reproducible record)
##with open("manc_v1p2_view_state.json", "w", encoding="utf-8") as f:
   ## json.dump(state, f, indent=2)

##print("Saved: manc_v1p2_view_state.json")

from pathlib import Path
import json

# Save outputs next to the notebook (or change OUTPUT_DIR to wherever you want)
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Save the exact CLIO URL you started from (provenance)
(OUTPUT_DIR / "clio_url.txt").write_text(CLIO_URL, encoding="utf-8")

# 2) Save the decoded Neuroglancer state JSON (reproducible record)
(OUTPUT_DIR / "manc_v1p2_view_state.json").write_text(
    json.dumps(state, indent=2),
    encoding="utf-8"
)

print("Saved:")
print(" -", OUTPUT_DIR / "clio_url.txt")
print(" -", OUTPUT_DIR / "manc_v1p2_view_state.json")





Top-level keys: ['title', 'dimensions', 'position', 'crossSectionOrientation', 'crossSectionScale', 'projectionOrientation', 'projectionScale', 'layers', 'showSlices', 'prefetch', 'selectedLayer', 'crossSectionBackgroundColor', 'projectionBackgroundColor', 'layout', 'selection', 'layerListPanel']
Number of layers: 28
Saved:
 - c:\Users\JHS\Documents\Python\project_folder_4B\output\clio_url.txt
 - c:\Users\JHS\Documents\Python\project_folder_4B\output\manc_v1p2_view_state.json


In [22]:
#make a csv file that describes the segmentation layers with my notes
##part 1: # Frist create a summary.df. This cell pulls out every segmentation layer’s name, segmentQuery, and the list of segment IDs it contains:

##import pandas as pd

layers = state.get("layers", [])
seg_layers = [L for L in layers if L.get("type") == "segmentation"]

# Build canonical dict
by_layer = {}
for L in seg_layers:
    name = L.get("name", "<unnamed>")
    body_ids = L.get("segments", []) or []

    by_layer[name] = {
        "n_neurons": len(body_ids),
        "segmentQuery": L.get("segmentQuery", None),
        "bodyIds": body_ids,  # full list
        "visible": L.get("visible", True),
        "tab": L.get("tab", None),
    }

# Print quick summary (top 15 largest groups)
summary_sorted = sorted(
    [(name, info["segmentQuery"], info["n_neurons"]) for name, info in by_layer.items()],
    key=lambda x: x[2],
    reverse=True,
)

for name, query, n in summary_sorted[:15]:
    print(f"{n:5d} neurons | {name} | query={query}")

# Build readable dataframe
summary_df = pd.DataFrame([
    {
        "layer": name,
        "n_neurons": info["n_neurons"],
        "visible": info["visible"],
        "tab": info["tab"],
        "segmentQuery": info["segmentQuery"],
        "bodyId_preview": ",".join(map(str, info["bodyIds"][:20])),
    }
    for name, info in by_layer.items()
]).sort_values("n_neurons", ascending=False).reset_index(drop=True)

summary_df

   97 neurons | manc-seg-v1.2-4B-all | query=#hemilineage:04B #somaNeuromere:T1 #somaSide:LHS
   73 neurons | manc-seg-v1.21- secondary n=73 | query=#hemilineage:04B #somaNeuromere:T1 #somaSide:LHS #birthtime:secondary
   35 neurons | nerves | query=None
   24 neurons | neuropils | query=None
   20 neurons | manc-seg-v1.22-4B-early secondary n= 20 | query=#hemilineage:04B #somaNeuromere:T1 #somaSide:LHS #birthtime:early_secondary
   10 neurons | manc-seg-v1.2-4B-subbclass BI n=4 | query=#hemilineage:04B #somaNeuromere:T1 #somaSide:LHS #subclass:BI
   10 neurons | manc-seg-v1.2 4B manual BR-2 n=10 | query=12523 162540 16527 10024 16487 17351 17409 18571 17534 101447
    7 neurons | court et al. tracts | query=None
    7 neurons | manc-seg-v1.2 4B manual  IR-nomorphology assigned-early secondary | query=20077 21567 17567 18156 101453 17572 18785
    7 neurons | manc-seg-v1.2 4B manual IR nomorphology assigned secondary | query=None
    6 neurons | manc-seg-v1.21-4B-subclassBR n=26 | quer

,layer,n_neurons,visible,tab,segmentQuery,bodyId_preview
0,manc-seg-v1.2-4B-all,97,True,segments,#hemilineage:04B #somaNeuromere:T1 #somaSide:LHS,"10024,100339,100525,101447,101453,102138,10253..."
1,manc-seg-v1.21- secondary n=73,73,False,rendering,#hemilineage:04B #somaNeuromere:T1 #somaSide:L...,"10024,100339,100525,101447,102138,102536,13128..."
2,nerves,35,False,segments,None,"1,10,11,12,13,14,15,16,17,18,19,2,20,21,22,23,..."
3,neuropils,24,True,segments,None,"10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,2..."
4,manc-seg-v1.22-4B-early secondary n= 20,20,False,rendering,#hemilineage:04B #somaNeuromere:T1 #somaSide:L...,"101453,10314,14389,15002,15309,155236,15811,15..."
5,manc-seg-v1.2 4B manual BR-2 n=10,10,False,rendering,12523 162540 16527 10024 16487 17351 17409 185...,"10024,101447,12523,162540,16487,16527,17351,17..."
6,manc-seg-v1.2-4B-subbclass BI n=4,10,False,rendering,#hemilineage:04B #somaNeuromere:T1 #somaSide:L...,"10314,11143,13436,14031,15393,15401,15969,1629..."
7,court et al. tracts,7,True,rendering,None,"100,101,102,103,104,105,106"
8,manc-seg-v1.2 4B manual IR nomorphology assign...,7,True,segments,None,"164146,18156,22868,23716,26941,27760,38071"
9,manc-seg-v1.2 4B manual IR-nomorphology assig...,7,False,segments,20077 21567 17567 18156 101453 17572 18785,"101453,17567,17572,18156,18785,20077,21567"


In [ ]:
# work layer by layer: 
# target_layer = "manc-seg-v1.2-4B-all"  # change this to whichever one you want
# info = by_layer.get(target_layer)

# if info is None:
#     raise ValueError(f"Layer not found. Available layers include:\n" + "\n".join(list(by_layer.keys())[:30]))

# ids = info["segments"]
# print("Layer:", target_layer)
# print("n_segments:", len(ids))
# print("segmentQuery:", info["segmentQuery"])
# print("first 30 IDs:", ids[:30])

In [23]:
# Step 2: add an empty  notes colun to summary.df. If someone views myURL and wonders why I grouped noeurons, these notes describe what the layer contains
##runnign this code will wipe out notes made previously, but they will be popualted again in the next cell.
summary_df = summary_df.copy()

# Add a column for your human explanations
summary_df["notes"] = ""

summary_df

#save notesto output folder
summary_df.to_csv(OUTPUT_DIR / "manc_layer_notes.csv", index=False)
print("Saved to:", OUTPUT_DIR / "manc_layer_notes.csv")

#load notes:
##summary_df = pd.read_csv(OUTPUT_DIR / "manc_layer_notes.csv")
##summary_df

Saved to: c:\Users\JHS\Documents\Python\project_folder_4B\output\manc_layer_notes.csv


In [24]:
#Step 3: Preparation for making my notes dictionary that will popualte the notes column of summary df. Dun this cell to print the segmentation layer names. Manually copy this output and add your notes betweenthe '' in the next cell
NOTES = {layer: "" for layer in summary_df["layer"]}

NOTES

{'manc-seg-v1.2-4B-all': '',
 'manc-seg-v1.21- secondary n=73': '',
 'nerves': '',
 'neuropils': '',
 'manc-seg-v1.22-4B-early secondary n= 20': '',
 'manc-seg-v1.2 4B manual BR-2 n=10': '',
 'manc-seg-v1.2-4B-subbclass BI n=4': '',
 'court et al. tracts': '',
 'manc-seg-v1.2 4B manual IR nomorphology assigned secondary': '',
 'manc-seg-v1.2 4B manual  IR-nomorphology assigned-early secondary': '',
 'manc-seg-v1.2 4B manual BR-1 n=6': '',
 'manc-seg-v1.2 4b-manual BR-4 n=6': '',
 'manc-seg-v1.21-4B-subclassBR n=26': '',
 'manc-seg-v1.2 4B manual IR-independent leg n=11': '',
 'manc-seg-v1.2 4B manual BR-3 n=4': '',
 'manc-seg-v1.2 4B- primary  n=4': '',
 'manc-seg-v1.22-4B subclass IA=4': '',
 'manc-seg-v1.2 4B manual  IR- proprioceptice tactile n=3': '',
 'manc:v1.2.1-TTMn': '',
 'all-synaptic': '',
 'all-tissue': '',
 'manc-seg-v1.2-4B-sublass IR n=60': '',
 'manc-seg-v1.21-4B subclass BA n=1': '',
 'voxel-classes': '',
 'initial-supervoxels': ''}

In [25]:
# Manually complete the notes in this cell in case required between the last before the  coma ''
NOTES = {
 'manc-seg-v1.2-4B-all': 'query: All 4B neurons selected by #hemilineage, soma side and thoracic level',
 'manc-seg-v1.21- secondary n=73': 'Secondary birthtime 4B neurons (query-derived)',
 'nerves': 'Anatomical context layer showing nerve ROIs',
 'neuropils': 'Neuropil segmentation for spatial orientation',
 'manc-seg-v1.22-4B-early secondary n= 20': 'Early secondary 4B neurons (query-derived)',
 'manc-seg-v1.2 4B manual BR-2 n=10': 'Manual morphology review group BR-2',
 'manc-seg-v1.2-4B-subbclass BI n=4': 'Subclass BI 4B neurons (query-derived)',
 'court et al. tracts': 'Published tract annotations (context only)',
 'manc-seg-v1.2 4B manual IR nomorphology assigned secondary': 'IR group, no morphology assigned, secondary, (filter query by birth order)',
 'manc-seg-v1.2 4B manual  IR-nomorphology assigned-early secondary': 'IR group, no morphology assigned, early secondary, ',
 'manc-seg-v1.2 4b-manual BR-4 n=6': 'Manual morphology group BR-4 (manual morphology review within query-derived BR group)',
 'manc-seg-v1.21-4B-subclassBR n=26': 'All BR subclass neurons, (query-derived)',
 'manc-seg-v1.2 4B manual IR-independent leg n=11': 'IR neurons independent of leg motor pools, (query-derived)',
 'manc-seg-v1.2 4B manual BR-3 n=4': 'Manual morphology group BR-3,(manual morphology review within query-derived BR group)',
 'manc-seg-v1.2 4B- primary  n=4': 'Primary birthtime 4B neurons, (filter query birth order)',
 'manc-seg-v1.22-4B subclass IA=4': 'Subclass IA neurons (query-derived)',
 'manc-seg-v1.2 4B manual  IR- proprioceptice tactile n=3': 'IR proprioceptive/tactile subtype, (manual morphology review filter by query subtype',
 'manc:v1.2.1-TTMn': 'TT motor neuron segmentation reference',
 'all-synaptic': 'All synaptic neuropil regions (context only)',
 'all-tissue': 'Whole tissue segmentation (context only)',
 'manc-seg-v1.2-4B-sublass IR n=60': 'All IR subclass neurons, (query-derived)',
 'manc-seg-v1.21-4B subclass BA n=1': 'BA subclass neuron, (query-derived)',
 'voxel-classes': 'Voxel classification layer (technical)',
 'initial-supervoxels': 'Initial segmentation supervoxels (technical)'
}

In [26]:
#complete the table: Fill out the summary notes column with the dictionary, but save it as manc_layer_notescsv.
summary_df["notes"] = summary_df["layer"].map(NOTES)
summary_df
summary_df.to_csv("manc_layer_notes.csv", index=False)
summary_df[["layer", "notes"]]
#overwrite previous table to append updated note column:
summary_df.to_csv(OUTPUT_DIR / "manc_layer_notes2.csv", index=False)
print("Saved to:", OUTPUT_DIR / "manc_layer_notes2.csv")

Saved to: c:\Users\JHS\Documents\Python\project_folder_4B\output\manc_layer_notes2.csv


In [29]:
# Build New State From Base + New Layers
import json
import urllib.parse
from copy import deepcopy
from pathlib import Path

OUTPUT_DIR = Path(OUTPUT_DIR)  # keep using your existing OUTPUT_DIR variable

# 1. Load base state fresh (do not trust in-memory state)
STATE_IN = OUTPUT_DIR / "manc_v1p2_view_state.json"  # starting state
with open(STATE_IN, "r", encoding="utf-8") as f:
    base_state = json.load(f)

state = deepcopy(base_state)

# 2. Define your new segmentation layers
TEST_LAYERS = {
    "manc-seg-v1.2 4B manual IR-1": [102536, 102138, 19833, 21189, 21322, 24181, 18253, 18629, 27760, 164146, 26941, 38071, 22868, 23716, 18156],
    "manc-seg-v1.2 4B manual IR-2": [22163, 100525, 153878, 21499, 18641, 29988, 165318, 22997, 18990],
    "manc-seg-v1.2 4B manual IR-3": [17935, 17567],
    "manc-seg-v1.2 4B manual IR-4": [20077, 21567, 101453, 17572, 18785],
}

# --- color palette (one color per layer, will cycle if more layers than colors) ---
# Sky blue, Orange, Vermillion, Pale violet
COLORS = ["#87CEEB", "#FFA500", "#E34234", "#CC99FF"]

# 3. Find the correct MANC segmentation source
seg_source = None
for L in state["layers"]:
    if L.get("type") == "segmentation":
        src = L.get("source")
        url = src.get("url") if isinstance(src, dict) else src
        if "manc-seg-v1p2/manc-seg-v1.2" in str(url):
            seg_source = src
            break

if seg_source is None:
    raise ValueError("Could not find MANC segmentation source.")

# 4. Append new layers, assigning a fixed color per layer
for i, (name, body_ids) in enumerate(TEST_LAYERS.items()):
    layer_color = COLORS[i % len(COLORS)]
    state["layers"].append({
        "type": "segmentation",
        "source": seg_source,
        "tab": "segments",
        "name": name,
        "segments": [str(int(b)) for b in body_ids],
        "segmentColors": {str(int(b)): layer_color for b in body_ids},
        "visible": True,
    })

# 5. Update title BEFORE saving and encoding
state["title"] = "4B analysis + manual IR layers"

# 6. Save new JSON (no spaces in filename)
STATE_OUT = OUTPUT_DIR / "manc_v1p2_with_manual_IR_layers.json"
with open(STATE_OUT, "w", encoding="utf-8") as f:
    json.dump(state, f, indent=2)

# 7. Generate URL from THIS state
encoded = urllib.parse.quote(json.dumps(state, separators=(",", ":")))
NEW_URL = "https://clio-ng.janelia.org/#!" + encoded

print("Saved new state:", STATE_OUT)
print("\nNew URL:\n")
print(NEW_URL)

Saved new state: c:\Users\JHS\Documents\Python\project_folder_4B\output\manc_v1p2_with_manual_IR_layers.json

New URL:

https://clio-ng.janelia.org/#!%7B%22title%22%3A%224B%20analysis%20%2B%20manual%20IR%20layers%22%2C%22dimensions%22%3A%7B%22x%22%3A%5B8e-09%2C%22m%22%5D%2C%22y%22%3A%5B8e-09%2C%22m%22%5D%2C%22z%22%3A%5B8e-09%2C%22m%22%5D%7D%2C%22position%22%3A%5B15353.5%2C35844.5%2C53248.5%5D%2C%22crossSectionOrientation%22%3A%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22%3A1%2C%22projectionOrientation%22%3A%5B-0.29617950320243835%2C0.698911726474762%2C-0.6356714367866516%2C0.14043490588665009%5D%2C%22projectionScale%22%3A26672.2540375702%2C%22layers%22%3A%5B%7B%22type%22%3A%22image%22%2C%22source%22%3A%7B%22url%22%3A%22precomputed%3A//gs%3A//flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22%3A%7B%22default%22%3Atrue%7D%2C%22enableDefaultSubsources%22%3Afalse%7D%2C%22tab%22%3A%22source%22%2C%22name%22%3A%22em